# Bell states and wire ordering
Explore Bell-state preparation and measurement.
Qiskit displays q0 on top and writes two-qubit basis labels as |q1 q0⟩. We put the left ket bit on q1. Measurements below use repeatable seeded sampling.

In [ ]:
%pip -q install "qiskit>=2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace
order = QuantumCircuit(2)
order.x(0)
assert np.allclose(Statevector.from_instruction(order).probabilities(), [0,1,0,0])
print("q0=1 gives", Statevector.from_instruction(order).probabilities_dict())
qc = QuantumCircuit(2)
qc.h(1); qc.cx(1, 0)
state = Statevector.from_instruction(qc)
assert np.allclose(state.probabilities(), [.5,0,0,.5])
a,b,c,d = state.data
assert np.isclose(abs(a*d-b*c), .5)
print(qc.draw(output="text"))

In [ ]:
state.seed(42)
print("Z-basis counts:", state.sample_counts(1024))
rho_local = partial_trace(state, [0])
assert np.allclose(rho_local.data, np.eye(2)/2)
print("One-qubit reduced state:", rho_local.data)

## Correlation is not enough
Compare the Bell state with a 50/50 classical mixture of |00⟩ and |11⟩. Both agree in Z measurements. After H on both qubits, their X-basis distributions differ.

In [ ]:
mix = DensityMatrix(np.diag([.5,0,0,.5]))
x_measure = QuantumCircuit(2)
x_measure.h(0); x_measure.h(1)
bell_x = state.evolve(x_measure).probabilities()
mix_x = mix.evolve(x_measure).probabilities()
assert np.allclose(bell_x, [.5,0,0,.5])
assert np.allclose(mix_x, [.25,.25,.25,.25])
print("Bell X probabilities:", bell_x)
print("Mixture X probabilities:", mix_x)

## Your turn
1. Add Z to one Bell qubit. Which basis detects the change?
2. Implement Q21 using q1 as the first (left) ket bit.
3. Explain why the reduced state is mixed even though the joint Bell state is pure.
Answer check: Z changes Bell + to Bell −; Z statistics stay the same and X correlations become opposite.